# Keeping a Secret on Someone Else's Machine, in a notebook

A runnable companion to the ilm.red engineering post. It builds every idea from the article
from scratch, in a few dozen lines: **envelope encryption**, **sealing a page** with authenticated
encryption (and catching tampering), an **access gate** that makes revocation real, a **faint
watermark**, and the **memory-versus-recall** trade.

It runs fully **offline** with a mock vault. No API keys, no accounts. Opens in Colab, runs top to bottom.

> The point is not production code. It is to make the moving parts small enough to hold in your hand.


## Setup

Two well-known libraries: `cryptography` (the sealing) and `Pillow` (the watermark demo).


In [ ]:
try:
    from cryptography.hazmat.primitives.ciphers.aead import AESGCM
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable,'-m','pip','install','-q','cryptography','Pillow'])
    from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os, base64
def b64(x): return base64.b64encode(x).decode()
print('ready')


## 1. Envelope encryption

Each book gets its own random **data key** (DEK). We never store that key in the clear: we lock it
inside one **master key** (KEK) that lives in a vault. Store the *wrapped* DEK next to the book. Steal
the database and you get locked boxes and locked-up keys, and nothing you can open.


In [ ]:
# The vault holds ONE master key. In real life it never leaves a hardened boundary.
KEK = AESGCM.generate_key(bit_length=256)

def make_dek():
    return AESGCM.generate_key(bit_length=256)   # a fresh per-book data key

def wrap(dek):
    iv = os.urandom(12)
    return iv + AESGCM(KEK).encrypt(iv, dek, None)   # DEK sealed by the KEK

def unwrap(blob):
    return AESGCM(KEK).decrypt(blob[:12], blob[12:], None)

dek = make_dek()
stored = wrap(dek)                      # <- this is what the database holds
print('wrapped DEK (what a thief steals):', b64(stored)[:48], '...')
print('unwraps back to the DEK:', unwrap(stored) == dek)


## 2. Seal a page (and catch tampering)

We seal each page with the DEK using AES-256-GCM. The stored blob is `IV || ciphertext`, and GCM
adds an authentication tag, so if even one byte is altered on disk, opening it fails loudly instead
of handing back quietly-corrupted data.


In [ ]:
def seal(dek, plaintext: bytes) -> bytes:
    iv = os.urandom(12)
    return iv + AESGCM(dek).encrypt(iv, plaintext, None)

def open_(dek, blob: bytes) -> bytes:
    return AESGCM(dek).decrypt(blob[:12], blob[12:], None)

page = b'Chapter 1. It was a bright cold day in April...'
blob = seal(dek, page)
print('at rest, the drawer holds nonsense:', b64(blob)[:48], '...')
print('opens correctly:', open_(dek, blob) == page)

# flip a single byte and watch it get rejected
tampered = bytearray(blob); tampered[20] ^= 1
try:
    open_(dek, bytes(tampered)); print('!! tamper NOT caught')
except Exception as e:
    print('tamper caught:', type(e).__name__)


## 3. The access gate (why revocation is real)

The only way to turn a wrapped DEK back into a usable one is one **key desk**. Before it unwraps
anything, it re-asks the question the reader already had to pass: *may you read this, right now?*
To take access away, you simply stop issuing. The key you never receive is a key you cannot use.


In [ ]:
VAULTED = {'origin-of-species': stored}     # book_id -> wrapped DEK
ACCESS  = {('alice','origin-of-species'): True, ('mallory','origin-of-species'): False}

def key_desk(viewer, book_id):
    if not ACCESS.get((viewer, book_id), False):
        raise PermissionError('forbidden')      # no key, ever
    return unwrap(VAULTED[book_id])

print('alice   ->', 'got a key' if key_desk('alice','origin-of-species') else '')
try:
    key_desk('mallory','origin-of-species')
except PermissionError as e:
    print('mallory ->', e)

# revoke alice: the door simply stops answering
ACCESS[('alice','origin-of-species')] = False
try:
    key_desk('alice','origin-of-species')
except PermissionError as e:
    print('alice (after revoke) ->', e)


## 4. A faint watermark

You cannot encrypt your way out of a camera. But you can **sign what the camera sees**: a faint,
per-reader mark, all but invisible while reading, present in any screenshot. Here it is on a sample page.


In [ ]:
from PIL import Image, ImageDraw, ImageFont
import math

def watermark(base: Image.Image, label: str, opacity=0.06) -> Image.Image:
    layer = Image.new('RGBA', base.size, (0,0,0,0))
    d = ImageDraw.Draw(layer)
    a = int(255*opacity)
    step_x, step_y = 220, 90
    for y in range(0, base.height, step_y):
        for x in range(0, base.width, step_x):
            d.text((x, y), label, fill=(20,20,20,a))
    return Image.alpha_composite(base.convert('RGBA'), layer)

page_img = Image.new('RGBA', (640,360), (250,247,238,255))
d = ImageDraw.Draw(page_img); d.text((40,40), 'On the Origin of Species', fill=(30,25,20,255))
marked = watermark(page_img, 'ilm.red - reader a1b2c3', opacity=0.06)
marked.convert('RGB').save('watermarked.png')
try:
    from IPython.display import Image as IPImg, display; display(IPImg('watermarked.png'))
except Exception:
    print('saved watermarked.png -- open it and squint; the mark is meant to be easy to miss')


## 5. Offline versus recall

To read on a plane, the key must be on the device. But a key on the device is one you cannot easily
recall. Two stances, and what each costs.


In [ ]:
# memory-only: the key vanishes when the tab closes; every resume re-checks access -> instant recall
# persisted:   the key is kept for offline reading -> reads with no signal, but recall is not instant
def can_read_offline(mode, has_signal, still_allowed):
    if has_signal:
        return still_allowed                 # both modes re-check when online
    return mode == 'persisted'               # only a kept key reads with no signal

for mode in ('memory-only','persisted'):
    for signal in (True, False):
        print(f'{mode:12} signal={signal!s:5} still_allowed=False -> reads={can_read_offline(mode, signal, False)}')
print('\nThe persisted key reads on the plane -- and keeps reading for a while even after access is pulled.')
print('That gap is the price of offline. Each book chooses where to stand.')


## That's the whole machine

Envelope-wrapped keys, sealed pages that catch tampering, one door that gates the key, a faint
signature for the leak no lock can stop, and an honest trade for offline. None of it makes a copy on
someone else's machine un-copyable. It moves the library from *leave it in plain view and hope* to
*seal it, hand out keys through one checked door, and know what you traded.*

Back to the essay: **Keeping a Secret on Someone Else's Machine** on the ilm.red engineering blog.
